# Contrastive direction

Representation engineering steers a model along the mean difference between its activations on positive and negative examples (Zou et al. 2023, "Representation Engineering: A Top-Down Approach to AI Transparency", arXiv:2310.01405; Rimsky et al. 2024, "Steering Llama 2 via Contrastive Activation Addition", arXiv:2312.06681). This notebook computes that direction at one site of a tiny classifier and adds it with `tl.steer` on a replay.

In [ ]:
# recipe-test: tool=torchlens expected=(1, 6)
import torch
from torch import nn

import torchlens as tl


class TinySentimentModel(nn.Module):
    """Tiny model with a readable activation direction."""

    def __init__(self) -> None:
        super().__init__()
        self.encoder = nn.Linear(6, 6, bias=False)
        self.head = nn.Linear(6, 2, bias=False)
        with torch.no_grad():
            self.encoder.weight.copy_(torch.eye(6))
            self.head.weight.zero_()
            self.head.weight[0, 0] = -1.0
            self.head.weight[1, 0] = 1.0

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(torch.relu(self.encoder(x)))


model = TinySentimentModel().eval()
positive_examples = torch.tensor([[2.0, 1.0, 0.0, 0.0, 0.0, 0.0], [1.5, 1.0, 0.0, 0.0, 0.0, 0.0]])
negative_examples = torch.tensor([[-2.0, 0.0, 0.0, 1.0, 0.0, 0.0], [-1.5, 0.0, 0.0, 1.0, 0.0, 0.0]])
neutral_example = torch.tensor([[0.1, 1.0, 0.0, 0.0, 0.0, 0.0]])
RESULT = tuple(neutral_example.shape)

Trace the positive and negative batches and read the same site from each. The direction is the mean positive activation minus the mean negative activation.

In [ ]:
# recipe-test: tool=torchlens expected=(6,)
positive_trace = tl.trace(model, positive_examples)
negative_trace = tl.trace(model, negative_examples)

positive_out = positive_trace.find_sites(tl.func("relu")).first().out.detach()
negative_out = negative_trace.find_sites(tl.func("relu")).first().out.detach()
direction = positive_out.mean(dim=0) - negative_out.mean(dim=0)
print(direction)
assert direction.shape == (6,)
RESULT = tuple(direction.shape)

Trace the neutral example and replay it with `tl.steer`. `dim=-1` names the feature axis.

In [ ]:
# recipe-test: tool=torchlens expected=True
base_trace = tl.trace(model, neutral_example)
steered_trace = base_trace.fork("positive_steering")
steered_trace.do(
    tl.func("relu"),
    tl.steer(direction, strength=0.5, dim=-1),
)

base_traceits = base_trace.output_ops[-1].out.detach()
steered_traceits = steered_trace.output_ops[-1].out.detach()
print("base logits", base_traceits)
print("steered logits", steered_traceits)
assert steered_traceits[0, 1] > base_traceits[0, 1]
RESULT = bool(steered_traceits[0, 1] > base_traceits[0, 1])